# 0. Unmodified pipeline peptides → optimal 99% isotope sets → sparse 1 Da bins

Use the pipeline's `dumped_peptides` artifact (`peptides.parquet`, column `peptide`),
produced by `ionmaidentools.pipelines.dump_peptides` and consumed by feature prediction.
Remove `[UNIMOD:<id>]` and numeric mass-shift tags (e.g. `[+15.9949]`) and terminal separators, then deduplicate bare sequences.
Target and decoy rows are both retained; charge states share the same neutral distribution.

`IsoTotalProb(0.99, get_minimal_pset=True)` selects the smallest set of fine-structure
isotopologues covering **at least 99%** probability. **After selection**, round absolute
neutral masses to the nearest integer Da (half up: `floor(mass + 0.5)`) and sum probabilities
within each occupied bin. This is nearest rounding, not mathematical ceiling. Include free
peptide termini (H₂O), with no modification mass, proton addition, or charge division.
Probabilities remain unconditional: the omitted tail is not renormalized.

Run in JupyterLab using the pipeline's `venvs/common/bin/python` kernel, which supplies
`IsoSpecPy`, `numpy`, `pandas`, `pandas_ops`, `mmappet`, and `pyarrow`. Run cells in order.
The first 1,000 source rows form a quick trial; set `MAX_PEPTIDE_ROWS = None` for all rows.
Reading the selected peptide column still loads the whole input before applying this limit.

In [3]:
from pathlib import Path
from tempfile import mkdtemp
import json
import re

import IsoSpecPy
import mmappet
import numpy as np
import pandas as pd
from pandas_ops.io import read_df
from IPython.display import display

# Works from the pipeline root, the IsoSpec root, or this notebooks directory.
PIPELINE_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "git" / "isospec").is_dir() and (p / "venvs" / "common").is_dir()
)
PEPTIDES_PATH = PIPELINE_ROOT / "results/f9468_dump_peptides_recreate/dumped_peptides/peptides.parquet"
MAX_PEPTIDE_ROWS = 1_000  # None processes every source row.
COVERAGE = 0.99
BATCH_SIZE = 1_000  # Unique sequences per mmappet append.
OUTPUT_PARENT = PIPELINE_ROOT / "git/isospec/notebooks/output"


ModuleNotFoundError: No module named 'IsoSpecPy'

## Strip modifications and retain source identity

`source_row` is the zero-based position in the input file, independent of its DataFrame index.
`sequence_id` follows first appearance of each bare sequence. Multiple modified peptides may
map to one bare sequence. Unknown or ambiguous residue symbols fail explicitly; U and O
(selenocysteine and pyrrolysine) are supported. The input's modified `monoisotopic` mass is
not used for the unmodified calculation.

In [ ]:
MODIFICATION_TAG = re.compile(r"\[(?:UNIMOD:\d+|[+-]\d+(?:\.\d+)?)\]", re.IGNORECASE)
BARE_PEPTIDE = re.compile(r"[ACDEFGHIKLMNPQRSTVWYOU]+")


def strip_modifications(peptide):
    if not isinstance(peptide, str):
        raise ValueError(f"Expected a peptide string, got {peptide!r}")
    sequence = MODIFICATION_TAG.sub("", peptide).strip("-")
    if BARE_PEPTIDE.fullmatch(sequence) is None:
        raise ValueError(f"Unsupported peptide notation or residues: {peptide!r}")
    return sequence


peptides = read_df(PEPTIDES_PATH, columns=["peptide"])
input_rows = len(peptides)
if MAX_PEPTIDE_ROWS is not None:
    if MAX_PEPTIDE_ROWS <= 0:
        raise ValueError("MAX_PEPTIDE_ROWS must be positive or None")
    peptides = peptides.iloc[:MAX_PEPTIDE_ROWS].copy()
if peptides.empty:
    raise ValueError("No peptide rows selected")
bare_sequences = peptides["peptide"].map(strip_modifications)
sequence_ids, unique_sequences = pd.factorize(bare_sequences, sort=False)
source_mapping = pd.DataFrame({
    "source_row": np.arange(len(peptides), dtype=np.uint64),
    "sequence_id": sequence_ids.astype(np.uint64),
})
sequences = pd.DataFrame({
    "sequence_id": np.arange(len(unique_sequences), dtype=np.uint64),
    "sequence": unique_sequences,
})
print(f"{input_rows:,} input rows; {len(peptides):,} selected; {len(sequences):,} bare sequences")
display(sequences.head())

## Select fine structure, then aggregate occupied bins

Only nonzero bins are stored. `mass_da` is an absolute neutral mass bin, not an isotope
number or an offset from the monoisotopic peak. Each unique sequence is calculated once.

In [ ]:
def aggregate_1da(masses, probabilities):
    rounded = np.floor(masses + 0.5).astype(np.int64)
    mass_da, inverse = np.unique(rounded, return_inverse=True)
    probability = np.bincount(inverse, weights=probabilities)
    return mass_da, probability


def isotope_bins(sequence):
    envelope = IsoSpecPy.IsoTotalProb(
        COVERAGE, fasta=sequence, formula="H2O", get_minimal_pset=True
    )
    probabilities = envelope.np_probs()
    mass_da, probability = aggregate_1da(envelope.np_masses(), probabilities)
    np.testing.assert_allclose(probability.sum(), probabilities.sum(), rtol=1e-13)
    assert COVERAGE - 1e-12 <= probability.sum() <= 1.0 + 1e-12
    return mass_da, probability


# Numerical checks: terminal/internal tags, rounding boundaries, probability
# aggregation, known PEPTIDE composition, and minimal-set coverage.
assert strip_modifications("M[+15.9949]GGGPGK") == "MGGGPGK"
assert strip_modifications("[UNIMOD:1]-AC[UNIMOD:4]DM[UNIMOD:35]K-[UNIMOD:2]") == "ACDMK"
for invalid in (None, "", "PEPXIDE", "M[Oxidation]K"):
    try:
        strip_modifications(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Unexpectedly accepted {invalid!r}")
masses, probabilities = aggregate_1da(
    np.array([100.49, 100.50, 101.49, 102.50]),
    np.array([0.1, 0.2, 0.3, 0.4]),
)
np.testing.assert_array_equal(masses, [100, 101, 103])
np.testing.assert_allclose(probabilities, [0.1, 0.5, 0.4])
reference = IsoSpecPy.IsoTotalProb(
    COVERAGE, formula="C34H53N7O15", get_minimal_pset=True
)
reference_mass, reference_prob = aggregate_1da(reference.np_masses(), reference.np_probs())
peptide_mass, peptide_prob = isotope_bins("PEPTIDE")
np.testing.assert_array_equal(peptide_mass, reference_mass)
np.testing.assert_allclose(peptide_prob, reference_prob, rtol=1e-12)
assert reference.np_probs().sum() - reference.np_probs().min() < COVERAGE
print("Sequence, rounding, composition, and coverage checks passed")

## Write long sparse mmappet tables

Each run gets a new directory under `notebooks/output/`, so rerunning cells cannot append
duplicate rows to a previous result. Output files:

- `isotopes.mmappet/`: `sequence_id:uint64`, `mass_da:int64`, `probability:float64`.
- `source_rows.mmappet/`: `source_row:uint64`, `sequence_id:uint64`.
- `sequences.tsv`: the `sequence_id` → bare sequence dictionary (mmappet stores numeric columns).
- `metadata.json`: input path/stat, row selection, coverage, rounding, and IsoSpec version.

Rows are ordered by sequence ID, then ascending mass. Batches bound the intermediate
isotope table's memory. A failed run may leave a partial directory; `metadata.json` is
written only after the export finishes.

In [ ]:
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path(mkdtemp(prefix="precursor_isotopes_99_", dir=OUTPUT_PARENT))

with mmappet.DatasetWriter(OUTPUT_DIR / "source_rows.mmappet") as writer:
    writer.append_df(source_mapping)
sequences.to_csv(OUTPUT_DIR / "sequences.tsv", sep="\t", index=False)

retained_probability = np.empty(len(sequences), dtype=np.float64)
bin_counts = np.empty(len(sequences), dtype=np.uint64)
with mmappet.DatasetWriter(OUTPUT_DIR / "isotopes.mmappet") as writer:
    for start in range(0, len(sequences), BATCH_SIZE):
        batch = []
        for sequence_id, sequence in sequences.iloc[start:start + BATCH_SIZE].itertuples(index=False, name=None):
            mass_da, probability = isotope_bins(sequence)
            retained_probability[sequence_id] = probability.sum()
            bin_counts[sequence_id] = len(mass_da)
            batch.append(pd.DataFrame({
                "sequence_id": np.full(len(mass_da), sequence_id, dtype=np.uint64),
                "mass_da": mass_da,
                "probability": probability,
            }))
        writer.append_df(pd.concat(batch, ignore_index=True))
        if start == 0 or (start // BATCH_SIZE + 1) % 100 == 0:
            print(f"Wrote {min(start + BATCH_SIZE, len(sequences)):,}/{len(sequences):,} sequences")

input_stat = PEPTIDES_PATH.stat()
metadata = {
    "input_path": str(PEPTIDES_PATH.resolve()),
    "input_size_bytes": input_stat.st_size,
    "input_mtime_ns": input_stat.st_mtime_ns,
    "input_rows": input_rows,
    "selected_rows": len(peptides),
    "unique_sequences": len(sequences),
    "isotope_rows": int(bin_counts.sum()),
    "coverage": COVERAGE,
    "get_minimal_pset": True,
    "rounding": "floor(neutral_mass_da + 0.5)",
    "modifications": "UNIMOD and numeric mass-shift tags removed; terminal H2O retained",
    "renormalized": False,
    "isospec_version": IsoSpecPy.__version__,
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(OUTPUT_DIR)

## Reopen and verify

Check persisted coverage and counts against the calculated values, then show the first
sequence's sparse envelope. Recover any input row's envelope by joining `source_rows` to
`isotopes` on `sequence_id`; source modification variants intentionally share an envelope.

In [ ]:
isotopes = mmappet.open_dataset(OUTPUT_DIR / "isotopes.mmappet")
persisted_mapping = mmappet.open_dataset(OUTPUT_DIR / "source_rows.mmappet")
pd.testing.assert_frame_equal(persisted_mapping, source_mapping)
assert len(isotopes) == int(bin_counts.sum())
assert list(isotopes.dtypes.astype(str)) == ["uint64", "int64", "float64"]
assert (isotopes["probability"] > 0).all()
assert not isotopes.duplicated(["sequence_id", "mass_da"]).any()
summary = isotopes.groupby("sequence_id", sort=True)["probability"].agg(["sum", "size"])
np.testing.assert_array_equal(summary.index, sequences["sequence_id"])
np.testing.assert_array_equal(summary["size"], bin_counts)
np.testing.assert_allclose(summary["sum"], retained_probability, rtol=1e-13)
assert summary["sum"].between(COVERAGE - 1e-12, 1 + 1e-12).all()
print(f"Round trip passed: {len(sequences):,} sequences → {len(isotopes):,} sparse rows")
display(isotopes.head(12))
display(summary.describe())